# 03 · Encoding Categorical Data

        > Part of the **Data Preprocessing** module — taught alongside the `data/customer_churn.csv` dataset.

        ## Learning objectives
        - Tell nominal from ordinal — and why the distinction changes the encoder
- Apply Label, Ordinal, and One-Hot encoding correctly
- Use `OneHotEncoder` from sklearn (the version that handles unseen categories)
- Handle high-cardinality columns with frequency / target encoding (and avoid leakage)
- Wire encoders into a `ColumnTransformer` pipeline

        ---

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

df = pd.read_csv("data/customer_churn.csv")
print("shape:", df.shape)
df.head()

shape: (5000, 15)


,customer_id,age,gender,city,education,tenure_months,contract_type,payment_method,monthly_charges,total_charges,num_products,has_credit_card,is_active_member,estimated_salary,churned
0,10000,46.0,Male,Bengaluru,Bachelors,40,Month-to-month,Electronic check,16.90,686.72,3,1,0,55083.68,1
1,10001,28.0,Female,Chennai,Masters,22,One year,Credit card,88.73,NaN,2,1,0,75325.39,0
2,10002,52.0,Female,Mumbai,Bachelors,7,Two year,Mailed check,67.45,449.61,1,1,1,57584.71,0
3,10003,54.0,Male,Delhi,Bachelors,31,One year,Bank transfer,48.61,1615.53,1,1,0,57525.63,0
4,10004,18.0,Male,Delhi,Bachelors,18,Month-to-month,Bank transfer,90.82,1612.81,1,1,0,103508.09,0


## 1. Nominal vs ordinal

- **Nominal** — categories with *no order*: `gender`, `city`, `payment_method`
- **Ordinal** — categories with a *meaningful order*: `education` (HS < Bachelors < Masters < PhD), `contract_type` (Month-to-month < One year < Two year)

Using a label encoder on a nominal column tells the model `Mumbai (0) < Delhi (1) < Bengaluru (2)` — which is nonsense and hurts most models.

In [3]:
df.select_dtypes(exclude="number").nunique()

gender            2
city              6
education         4
contract_type     3
payment_method    4
dtype: int64

## 2. Label Encoding — for the **target** or true ordinal columns

`LabelEncoder` was designed for **the target column `y`**, not for features. For features
use `OrdinalEncoder` instead — it accepts a 2D array and lets you specify the order.

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(df["gender"])
print("classes:", le.classes_)
print("first 10:", y_enc[:10])

classes: ['Female' 'Male']
first 10: [1 0 0 1 1 1 1 1 0 1]


## 3. Ordinal Encoding — for ordered categories with a known order

Always pass `categories=` explicitly. The default alphabetic order will silently
miscode `education` as `Bachelors=0, High School=1, Masters=2, PhD=3`.

In [5]:
from sklearn.preprocessing import OrdinalEncoder

education_order = ["High School", "Bachelors", "Masters", "PhD"]
contract_order  = ["Month-to-month", "One year", "Two year"]

ord_enc = OrdinalEncoder(
    categories=[education_order, contract_order],
    handle_unknown="use_encoded_value", unknown_value=-1,
)
sample = df[["education", "contract_type"]].dropna().head(8)
ord_enc.fit_transform(sample)

array([[1., 0.],
       [2., 1.],
       [1., 2.],
       [1., 1.],
       [1., 0.],
       [2., 0.],
       [0., 1.],
       [0., 2.]])

## 4. One-Hot Encoding — for nominal columns

Creates one binary column per category. The right default for nominal features fed to
linear / distance / neural models. Tree models work fine with it too.

Two common APIs:
- `pd.get_dummies()` — quick and DataFrame-friendly
- `sklearn.preprocessing.OneHotEncoder` — the production choice (handles unseen categories at inference time, plays nicely with pipelines)

In [6]:
pd.get_dummies(df["payment_method"], prefix="pay").head()

,pay_Bank transfer,pay_Credit card,pay_Electronic check,pay_Mailed check
0,False,False,True,False
1,False,True,False,False
2,False,False,False,True
3,True,False,False,False
4,True,False,False,False


In [7]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = ohe.fit_transform(df[["gender", "city", "payment_method"]])
pd.DataFrame(encoded, columns=ohe.get_feature_names_out()).head()

,gender_Female,gender_Male,city_Bengaluru,city_Chennai,city_Delhi,city_Hyderabad,city_Mumbai,city_Pune,payment_method_Bank transfer,payment_method_Credit card,payment_method_Electronic check,payment_method_Mailed check
0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### `drop="first"` — when and why

For linear models with an intercept, the *k* one-hot columns are perfectly collinear
(their sum is always 1). Drop one to avoid the **dummy variable trap**. For tree-based
models, leaving all *k* in is fine and slightly more interpretable.

In [8]:
ohe_drop = OneHotEncoder(drop="first", sparse_output=False)
ohe_drop.fit_transform(df[["gender", "contract_type"]])[:5]

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 1., 0.],
       [1., 0., 0.]])

## 5. High-cardinality columns

One-hot a column with 10,000 unique values and you have 10,000 new columns. Two
common alternatives:

### Frequency encoding
Replace each category with how often it appears. Simple, no leakage.

In [9]:
freq = df["city"].value_counts(normalize=True)
df["city_freq"] = df["city"].map(freq)
df[["city", "city_freq"]].head()

,city,city_freq
0,Bengaluru,0.1808
1,Chennai,0.1232
2,Mumbai,0.2488
3,Delhi,0.2298
4,Delhi,0.2298


### Target (mean) encoding — *with leakage warning*

Replace each category with the mean of `y` for that category. Powerful but dangerous:
if you compute the means on the full dataset, you've leaked the target into your features.

**Always compute means on the training fold only**, ideally with cross-validated smoothing.

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

target_mean = y_train.groupby(X_train["city"]).mean()
global_mean = y_train.mean()

X_train["city_te"] = X_train["city"].map(target_mean)
X_test["city_te"]  = X_test["city"].map(target_mean).fillna(global_mean)

X_train[["city", "city_te"]].drop_duplicates().head()

,city,city_te
2638,Mumbai,0.101420
2997,Pune,0.095745
3342,Chennai,0.058233
3433,Bengaluru,0.089163
2316,Delhi,0.104701


## 6. Putting it together — `ColumnTransformer`

Real pipelines apply different encoders to different columns. `ColumnTransformer` is
the clean way to express that.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

nominal_cols = ["gender", "city", "payment_method"]
ordinal_cols = ["education", "contract_type"]
numeric_cols = ["age", "tenure_months", "monthly_charges",
                "total_charges", "estimated_salary",
                "num_products", "has_credit_card", "is_active_member"]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])

nominal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe",    OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord",    OrdinalEncoder(
        categories=[education_order, contract_order],
        handle_unknown="use_encoded_value", unknown_value=-1)),
])

pre = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("nom", nominal_pipe, nominal_cols),
    ("ord", ordinal_pipe, ordinal_cols),
])

X_proc = pre.fit_transform(X_train.drop(columns=["city_te"]))
print("Processed shape:", X_proc.shape)

Processed shape: (4000, 22)


## 7. Cheat sheet

| Column kind | Encoder |
|-------------|---------|
| Target / true ordinal | `LabelEncoder` (target) or `OrdinalEncoder` (features) |
| Nominal, low cardinality (<~15) | `OneHotEncoder` |
| Nominal, high cardinality | frequency or target encoding |
| Linear model with intercept | `OneHotEncoder(drop="first")` |
| Tree model | `OneHotEncoder()` (no drop) |

## Exercise
1. Why does using `LabelEncoder` on `city` produce *worse* results than one-hot for
   a logistic regression, but make little difference for a random forest?
2. Implement target encoding using *5-fold out-of-fold* means (no leakage). Hint:
   `KFold` + `groupby` inside the loop.